# WaterOnt: convert your EPANET model for the explorer

Turns your EPANET model (`.inp`) into a network file (`.json`) that the [WaterOnt explorer](https://manuelhf.github.io/WaterOnt/) opens with **Open network file (.json)…**. No Python knowledge needed.

**How to use**
1. Run step 1 (installs WaterOnt; about a minute). If the runtime restarts, just run step 1 again.
2. Run step 2 and, when the **Choose Files** button appears, select your `.inp` and, optionally, any of these CSV files (in one go):
   - an **asset register** or **inspection log** (a pipe column plus columns such as installation year, material, condition, last inspection; one row per inspection is fine, the latest wins),
   - a **failure list** (pipe and date of each failure),
   - a **sensor layout** (element and sensor type),
   - **critical customers** (junction and type, e.g. Hospital).

   Column names need not match exactly ("Pipe ID", "Last inspection", "Installation year" all work), dates may be used for years, and Excel files saved as CSV with commas or semicolons both work. Templates: [`data/`](https://github.com/manuelhf/WaterOnt/tree/main/data).
3. The `.json` downloads automatically. Open it in the explorer.

Your files are processed in this Colab session only; they are not stored or shared.

In [ ]:
# Step 1: install WaterOnt (keeps NumPy compatible with Colab's SciPy)
import os, subprocess, sys
os.chdir("/content")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy>=2.0,<2.3",
                "git+https://github.com/manuelhf/WaterOnt.git"], check=True)
on_disk = subprocess.run([sys.executable, "-c", "import numpy; print(numpy.__version__)"],
                         capture_output=True, text=True).stdout.strip()
import numpy
if numpy.__version__ != on_disk:
    print("Restarting to load the installed packages. When it reconnects, run step 1 again.")
    os.kill(os.getpid(), 9)
try:
    from wateront.convert import convert
    print("Step 1 done: WaterOnt installed. Now run step 2.")
except ImportError:
    print("The WaterOnt version on GitHub is too old for this converter: update the repository and run step 1 again.")

In [ ]:
# Step 2: upload your files, convert, download the network file
import os, shutil
from google.colab import files
from wateront.convert import convert, classify

work = "/content/wateront_upload"
shutil.rmtree(work, ignore_errors=True); os.makedirs(work)
os.chdir(work)
uploaded = files.upload()                       # select the .inp and any CSV files together
paths = [os.path.join(work, name) for name in uploaded]
for p in paths:
    print(f"  {os.path.basename(p)}: {classify(p)}")
try:
    out, summary = convert(paths, out_dir=work)
except ValueError as err:
    print("\nNothing was converted:", err)
else:
    print()
    print("\n".join(summary))
    files.download(out)
    print("\nOpen the downloaded file at https://manuelhf.github.io/WaterOnt/ with \"Open network file (.json)…\".")